# SoundStream Demo

В этом ноутбуке показано, как можно применить предобученный SoundStream codec для сжатия аудиофайлов.

In [ ]:
!pip install wget==3.2

In [ ]:
from pathlib import Path
from urllib.parse import urlparse
import wget
from IPython.display import Audio, display

In [ ]:
AUDIO_URL = "https://keithito.com/LJ-Speech-Dataset/LJ025-0076.wav"

## Клонирование репозитория

**Ячейку ниже нужно запускать, только если ноутбук скачан отдельно, без репозитория**

Произойдёт клонирование репозитория с моделью в текущую директорию.

In [ ]:
!git clone https://github.com/Zagonov/dlr-soundstream.git
%cd dlr-soundstream

## Установка необходимых библиотек

Устанавливает необходимые библиотеки из `requirements.txt` репозитория.

In [ ]:
# эта ячейка у меня в колабе отрабатывает ~4 минуты
# так как он решает какие-то там несовместимости свои
# но в итоге всё работает
# очень прошу подождать
!pip install -q -r requirements.txt

## Скачивание чекпоинта

Если `saved/model_best.pth` уже существует локально, используется он.

Иначе чекпоинт скачивается с HuggingFace.

In [ ]:
checkpoint_path = Path("saved/model_best.pth")
checkpoint_path.parent.mkdir(exist_ok=True)
checkpoint_url = "https://huggingface.co/Lunfus/soundstream/resolve/main/model_best.pth"

if not checkpoint_path.exists():
    wget.download(checkpoint_url, str(checkpoint_path))

## Скачивание кастомного аудиофайла

Скачивает файл по ссылке `AUDIO_URL`.

In [ ]:
examples_dir = Path("examples")
examples_dir.mkdir(parents=True, exist_ok=True)
filename = Path(urlparse(AUDIO_URL).path).name
input_path = examples_dir / filename
wget.download(AUDIO_URL, str(input_path))

## Запуск кодека


Прогоняет скачанный файл через codec при помощи `inference.py` из корня репозитория. Внутри происходит создание модели из `baseline` конфига, подгрузка в неё чекпоинта и прогон модели.

Возвращается словарь `result`, в котором содержится оригинальное аудио, реконструированное, sample rate, индексы кодбука и путь к сгенерированному аудио. Также реконструированное аудио сохраняется в `output_path`

In [ ]:
from inference import run_codec

In [ ]:
result = run_codec(
    input_path=input_path,
    checkpoint_path=checkpoint_path,
    output_path=examples_dir / "reconstructed.wav"
)

original_audio = result["original_audio"]
reconstructed_audio = result["reconstructed_audio"]
sr = result["sample_rate"]
indices = result["indices"]

## Прослушивание результата

Два плейера: первый - оригинальное аудио, второй - реконструированное.

In [ ]:
print("Original audio")
display(Audio(original_audio.numpy(), rate=sr))

print("Reconstructed audio")
display(Audio(reconstructed_audio.numpy(), rate=sr))